In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(
    "../data/raw/accepted_2007_to_2018Q4.csv",
    nrows=50000,
    low_memory=False
)

df = df[
    df["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )
].copy()

df["default"] = df["loan_status"].apply(
    lambda x: 0 if x == "Fully Paid" else 1
)

df.shape

(44006, 152)

In [4]:
df["avg_fico"] = (
    df["fico_range_low"]
    + df["fico_range_high"]
) / 2

In [5]:
df[
    [
        "fico_range_low",
        "fico_range_high",
        "avg_fico"
    ]
].head()

,fico_range_low,fico_range_high,avg_fico
0,675.0,679.0,677.0
1,715.0,719.0,717.0
2,695.0,699.0,697.0
4,695.0,699.0,697.0
5,690.0,694.0,692.0


In [6]:
def convert_emp_length(x):

    if pd.isna(x):
        return np.nan

    if x == "< 1 year":
        return 0

    if x == "10+ years":
        return 10

    return int(x.split()[0])

In [7]:
df["emp_length_num"] = (
    df["emp_length"]
    .apply(convert_emp_length)
)

In [8]:
df[
    ["emp_length", "emp_length_num"]
].head(10)

,emp_length,emp_length_num
0,10+ years,10.0
1,10+ years,10.0
2,10+ years,10.0
4,3 years,3.0
5,4 years,4.0
6,10+ years,10.0
7,10+ years,10.0
8,6 years,6.0
9,10+ years,10.0
12,3 years,3.0


In [9]:
df["income_bucket"] = pd.cut(
    df["annual_inc"],
    bins=[0, 50000, 100000, np.inf],
    labels=["Low", "Medium", "High"]
)

In [10]:
df["income_bucket"].value_counts()

income_bucket
Medium    21635
Low       13626
High       8744
Name: count, dtype: int64

In [11]:
df["dti_risk"] = pd.cut(
    df["dti"],
    bins=[0, 15, 30, np.inf],
    labels=["Low", "Medium", "High"]
)

In [12]:
df["dti_risk"].value_counts()

dti_risk
Medium    23070
Low       15526
High       5396
Name: count, dtype: int64

In [13]:
df[
    [
        "dti",
        "emp_length_num",
        "avg_fico"
    ]
].isnull().sum()

dti                  1
emp_length_num    2757
avg_fico             0
dtype: int64

In [14]:
df["dti"] = df["dti"].fillna(
    df["dti"].median()
)

df["emp_length_num"] = df["emp_length_num"].fillna(
    df["emp_length_num"].median()
)

In [15]:
df[
    [
        "dti",
        "emp_length_num",
        "avg_fico"
    ]
].isnull().sum()

dti               0
emp_length_num    0
avg_fico          0
dtype: int64

In [16]:
model_df = df[
    [
        "default",
        "loan_amnt",
        "annual_inc",
        "dti",
        "int_rate",
        "installment",
        "avg_fico",
        "emp_length_num",
        "home_ownership",
        "purpose",
        "grade"
    ]
].copy()

In [17]:
model_df.shape

(44006, 11)

In [18]:
[
    "loan_amnt",
    "annual_inc",
    "dti",
    "int_rate",
    "installment",
    "avg_fico",
    "emp_length_num",
    "home_ownership",
    "purpose",
    "grade"
]

['loan_amnt',
 'annual_inc',
 'dti',
 'int_rate',
 'installment',
 'avg_fico',
 'emp_length_num',
 'home_ownership',
 'purpose',
 'grade']

In [19]:
model_df = df[
    [
        "default",
        "loan_amnt",
        "annual_inc",
        "dti",
        "int_rate",
        "installment",
        "avg_fico",
        "emp_length_num",
        "home_ownership",
        "purpose",
        "grade"
    ]
].copy()

In [20]:
model_df.shape

(44006, 11)

In [21]:
model_df = pd.get_dummies(
    model_df,
    columns=[
        "home_ownership",
        "purpose",
        "grade"
    ],
    drop_first=True
)

In [22]:
model_df.shape

(44006, 28)

In [23]:
model_df.head()

,default,loan_amnt,annual_inc,dti,int_rate,installment,avg_fico,emp_length_num,home_ownership_MORTGAGE,home_ownership_OWN,...,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G
0,0,3600.0,55000.0,5.91,13.99,123.03,677.0,10.0,True,False,...,False,False,False,False,False,True,False,False,False,False
1,0,24700.0,65000.0,16.06,11.99,820.28,717.0,10.0,True,False,...,False,False,True,False,False,True,False,False,False,False
2,0,20000.0,63000.0,10.78,10.78,432.66,697.0,10.0,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,10400.0,104433.0,25.37,22.45,289.91,697.0,3.0,True,False,...,False,False,False,False,False,False,False,False,True,False
5,0,11950.0,34000.0,10.20,13.44,405.18,692.0,4.0,False,False,...,False,False,False,False,False,True,False,False,False,False


In [24]:
model_df.info()

<class 'pandas.DataFrame'>
Index: 44006 entries, 0 to 49999
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   default                     44006 non-null  int64  
 1   loan_amnt                   44006 non-null  float64
 2   annual_inc                  44006 non-null  float64
 3   dti                         44006 non-null  float64
 4   int_rate                    44006 non-null  float64
 5   installment                 44006 non-null  float64
 6   avg_fico                    44006 non-null  float64
 7   emp_length_num              44006 non-null  float64
 8   home_ownership_MORTGAGE     44006 non-null  bool   
 9   home_ownership_OWN          44006 non-null  bool   
 10  home_ownership_RENT         44006 non-null  bool   
 11  purpose_credit_card         44006 non-null  bool   
 12  purpose_debt_consolidation  44006 non-null  bool   
 13  purpose_home_improvement    44006 non-null  boo

In [25]:
model_df.to_csv(
    "../data/processed/credit_risk_model_data.csv",
    index=False
)

In [26]:
import os

os.path.exists(
    "../data/processed/credit_risk_model_data.csv"
)

True